In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks/practice")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


    # 04 · The runtime policy enforcement point — practice

    **Primer section:** §4. Evaluate requests against `policies/support-agent.yaml`, dry-run a plan,
    then drive the ADK loop through a confirmation.

> **How to use this practice notebook.** Every `____` is a blank you must fill (a name, an
> argument, an expression); a `raise NotImplementedError("fill me")` means "write the body".
> Each exercise ends with `assert` checks — run the cell, and if it is silent you got it right.
> The completed version lives in `notebooks/solutions/`.

In [ ]:
import logging

from agentsec.logging_utils import quiet_logs

quiet_logs(logging.ERROR)

from agentsec.agents import REFUNDS, LocalStack, Step, reset_demo_state
from agentsec.audit import AuditLog
from agentsec.config import Settings
from agentsec.identity import AgentIdentity, AuthorityContext, UserPrincipal
from agentsec.policy import Effect, Policy, PolicyEngine, ToolCallRequest
from agentsec.runtime import confirm, run_turn, seed_session

reset_demo_state()

settings = Settings()
agent = settings.agent_identity()
ORG, PROJECT = settings.org_id, settings.project_number
other_project_agent = AgentIdentity.for_agent_engine(project_number="111111111111", location="us-central1", engine_id="marketing-agent", org_id=ORG)
sibling_agent = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="billing-agent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

policy = Policy.from_yaml(settings.policy_path)
engine = PolicyEngine(policy)

def req(tool, authority, **args):
    return ToolCallRequest(tool=tool, args=args, authority=authority)

def show(label, decision):
    print(f"{label:<58} {decision.effect.value.upper():<8} {'; '.join(decision.reasons) or '-'}")

## Exercise 1 — build the authority contexts and hit the deny reasons

Create a delegated context for Ana with the scopes below and an own-authority context, then check
the effect and the first reason of each request.

In [ ]:
delegated = AuthorityContext.____(agent, ana, {"customers:read", "orders:read", "payments:refund", "email:send"})
own = AuthorityContext.____(agent, {"customers:read"})

d = engine.evaluate(req("run_sql", delegated, query="select 1"))
assert d.effect is Effect.DENY and "default deny" in d.reasons[0]

d = engine.evaluate(req("lookup_customer", AuthorityContext.delegated(other_project_agent, ana, {"customers:read"}), email="x"))
assert d.effect is Effect.DENY and "not in allow list" in d.reasons[0]

d = engine.evaluate(req("lookup_customer", own, email="x"))
assert d.effect is Effect.DENY and "delegated" in d.reasons[0]

d = engine.evaluate(req("lookup_customer", AuthorityContext.delegated(agent, ana, set()), email="x"))
assert d.effect is Effect.DENY and "missing scopes" in d.reasons[0]

assert engine.evaluate(req("search_knowledge", own, query="refund")).allowed
assert engine.evaluate(req("lookup_customer", delegated, email="x")).allowed
print("deny reasons: unknown tool, principal, authority, scopes — all distinct")

## Exercise 2 — predict the refund decisions

Fill in the expected effect for each refund request (`Effect.ALLOW`, `Effect.CONFIRM` or
`Effect.DENY`). Read the policy's constraints and `unless` expression first.

In [ ]:
def refund(**a):
    return req("issue_refund", delegated, **{"order_id": "O-5001", "currency": "USD", "reason": "dup", **a})

expected = {
    "35 USD": Effect.____,
    "120 USD": Effect.____,
    "35 SGD": Effect.____,
    "5000 USD": Effect.____,
    "10 EUR": Effect.____,
}
actual = {
    "35 USD": engine.evaluate(refund(amount=35)).effect,
    "120 USD": engine.evaluate(refund(amount=120)).effect,
    "35 SGD": engine.evaluate(refund(amount=35, currency="SGD")).effect,
    "5000 USD": engine.evaluate(refund(amount=5000)).effect,
    "10 EUR": engine.evaluate(refund(amount=10, currency="EUR")).effect,
}
assert actual == expected, actual
approved = ToolCallRequest(tool="issue_refund", args={"order_id": "O-5001", "amount": 120, "currency": "USD", "reason": "x"}, authority=delegated, confirmed_by="ana@customer.example")
assert engine.evaluate(approved).allowed
print({k: v.value for k, v in actual.items()})

## Exercise 3 — egress and a dry run

List which of the URLs `fetch_url` may reach, then dry-run the plan and find the first step that the
destructive-call budget (2 per invocation) denies.

In [ ]:
urls = ["https://docs.acme.example/refunds", "http://docs.acme.example/x", "https://evil.example/x",
        "https://169.254.169.254/computeMetadata/v1/", "https://docs.acme.example.evil.example/", "https://storage.googleapis.com/b/o"]
allowed_urls = [u for u in urls if engine.evaluate(req("fetch_url", own, url=u)).allowed]
assert allowed_urls == ["https://docs.acme.example/refunds", "https://storage.googleapis.com/b/o"]

plan = [
    ("lookup_customer", {"email": "ana@customer.example"}),
    ("issue_refund", {"order_id": "O-5001", "amount": 10, "currency": "USD", "reason": "a"}),
    ("issue_refund", {"order_id": "O-5002", "amount": 10, "currency": "USD", "reason": "b"}),
    ("issue_refund", {"order_id": "O-5003", "amount": 10, "currency": "USD", "reason": "c"}),
]
decisions = engine.____(plan, delegated)
first_denied = next(i for i, d in enumerate(decisions) if d.effect is Effect.____)
assert first_denied == 3 and "budget exceeded" in decisions[3].reasons[0]
print("allowed:", allowed_urls)
print("first denied step:", plan[first_denied][0], "-", decisions[first_denied].reasons[0])

## Exercise 4 — the ADK loop: default deny, envelope, confirmation

Seed a session for Ana with the scopes, script the model to call `run_sql`, a 35 USD refund on
`O-5002` and a 120 USD refund on `O-5001`, run one turn, then **approve** the pending confirmation.

In [ ]:
stack = LocalStack.create(Settings(), audit=AuditLog())
USER = {"subject": "u-ana", "email": "ana@customer.example", "tenant": "acme"}
SCOPES = ["customers:read", "orders:read", "payments:refund", "email:send"]
await seed_session(stack.runner, user_id="u-ana", session_id="s1", user=____, scopes=____)

stack.script(
    Step.call("run_sql", query="select 1"),
    Step.call("issue_refund", order_id="O-5002", amount=35.0, currency="USD", reason="dup"),
    Step.call("issue_refund", order_id="O-5001", amount=120.0, currency="USD", reason="cancelled"),
    Step.say("done"),
)
r = await run_turn(stack.runner, user_id="u-ana", session_id="s1", message="refund my orders")
by_name = {t["name"]: t["response"] for t in r.tool_responses}
assert by_name["run_sql"]["error"] == "policy_denied"
assert len(REFUNDS) == 1 and REFUNDS[0]["amount"] == 35.0
assert len(r.pending_confirmations) == 1

p = r.pending_confirmations[0]
assert p.original_call["args"]["amount"] == 120.0 and p.payload["authority"]["user"] == "ana@customer.example"
stack.script(Step.say("Refund issued."))
r_ok = await confirm(stack.runner, user_id="u-ana", session_id="s1", pending=____, confirmed=____)
assert len(REFUNDS) == 2 and REFUNDS[-1]["amount"] == 120.0
print(r_ok.summary())

## Exercise 5 — reject, and read the audit trail

In a new session, script a 480 SGD refund on `O-5003`, reject the confirmation, and prove nothing
executed. Then find the approver recorded for the earlier approval.

In [ ]:
await seed_session(stack.runner, user_id="u-ana", session_id="s2", user=USER, scopes=SCOPES)
stack.script(Step.call("issue_refund", order_id="O-5003", amount=480.0, currency="SGD", reason="request"), Step.say("ok"))
r2 = await run_turn(stack.runner, user_id="u-ana", session_id="s2", message="refund O-5003")
stack.script(Step.say("Nothing was refunded."))
r2_no = await confirm(stack.runner, user_id="u-ana", session_id="s2", pending=r2.pending_confirmations[0], confirmed=____)
assert any(t["response"].get("error") == "policy_denied" for t in r2_no.tool_responses)
assert len(REFUNDS) == 2

approvers = [e.approver for e in stack.audit.events() if e.____]
assert approvers == ["ana@customer.example"]
denied_tools = {e.tool for e in stack.audit.____()}
assert {"run_sql", "issue_refund"} <= denied_tools
for row in stack.audit.timeline():
    print(row)

**In one sentence:** "Deny by default in a policy file, enforced in the runtime callback that
sees the actual tool call; destructive tools need confirmation unless a narrow argument envelope
holds; the confirmation shows the real arguments; and every decision leaves an event with both
identities and the approver."